# Clasificador de Voz — Embeddings wav2vec2
## Enojo, Tristeza y Feliz · Dataset filtrado (10 por clase)

Usa **facebook/wav2vec2-base** como extractor de embeddings de 768 dimensiones.

| Parámetro | Valor |
|---|---|
| Modelo | `facebook/wav2vec2-base` |
| Sample rate | 16 kHz |
| Embedding | 768 dims (mean-pooling temporal) |
| Dataset | 10 audios por clase |
| Evaluación | LOOCV |


In [ ]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_DIR  = os.path.join('..', 'data', 'AUDIOS_FILTRADOS_V2')
REPORTE   = os.path.join('..', 'outputs', 'reporte_filtrado_v2.csv')
CLASES    = ['Enojo', 'Tristeza', 'Feliz']
COLORES   = {'Enojo': '#DD8452', 'Tristeza': '#4C72B0', 'Feliz': '#E377C2'}
SCORE_MAP = {'Enojo': 'score_enojo', 'Tristeza': 'score_tristeza', 'Feliz': 'score_feliz'}
TARGET_SR = 16000
CACHE     = os.path.join('..', 'outputs', 'embeddings_wav2vec2.npz')
N_PER_CLASS = 10

DEVICE = 'mps'  if torch.backends.mps.is_available() else          'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'Clases activas: {CLASES}')


---
## 2. Cargar modelo wav2vec2

La primera ejecución descarga ~95 MB desde Hugging Face Hub.  
Las siguientes usan el caché local automáticamente.

In [ ]:
print('Cargando facebook/wav2vec2-base...')
t0 = time.time()
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base')
model     = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(DEVICE)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo listo en {time.time()-t0:.1f}s — {n_params/1e6:.1f}M parámetros')
print(f'Hidden size: {model.config.hidden_size} dims')

---
## 3. Extracción de embeddings

Para cada audio: cargar → resamplear a 16kHz → wav2vec2 → mean-pooling → vector 768-dim.  
Si existe el caché , lo reutiliza sin recalcular.

In [ ]:
def get_embedding(ruta):
    try:
        y, _ = librosa.load(ruta, sr=TARGET_SR, mono=True, duration=10.0)
        if len(y) < TARGET_SR * 0.3: return None
        inputs = processor(y, sampling_rate=TARGET_SR, return_tensors='pt', padding=True)
        with torch.no_grad():
            out = model(inputs.input_values.to(DEVICE))
        return out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().astype(np.float32)
    except Exception as e:
        print(f'  [ERROR] {os.path.basename(ruta)}: {e}')
        return None

df_rep = pd.read_csv(REPORTE)
seleccion_por_clase = {}
for clase in CLASES:
    score_col = SCORE_MAP[clase]
    carpeta = os.path.join(DATA_DIR, clase)
    seleccion = (
        df_rep[
            (df_rep['clase_original'] == clase)
            & df_rep['archivo'].apply(lambda x: os.path.exists(os.path.join(carpeta, x)))
        ]
        .sort_values(score_col, ascending=False)
        .head(N_PER_CLASS)['archivo']
        .tolist()
    )
    seleccion_por_clase[clase] = seleccion
    print(f'{clase:<9} ({len(seleccion)}): {seleccion}')

if os.path.exists(CACHE):
    print(f'Cargando embeddings desde caché: {CACHE}')
    c = np.load(CACHE, allow_pickle=True)
    X, y, rec = c['X'], c['y'], c['rec']
    esperadas = N_PER_CLASS * len(CLASES)
    if len(y) != esperadas:
        print('  Caché tiene tamaño distinto, recalculando...')
        os.remove(CACHE)

if not os.path.exists(CACHE):
    print('Extrayendo embeddings...')
    regs = []
    plan = [(archivo, clase) for clase in CLASES for archivo in seleccion_por_clase[clase]]
    for i, (archivo, clase) in enumerate(plan, 1):
        ruta = os.path.join(DATA_DIR, clase, archivo)
        print(f'  [{i:2d}/{len(plan)}] {clase}/{archivo}', flush=True)
        emb = get_embedding(ruta)
        if emb is not None:
            regs.append({'etiqueta': clase, 'rec': archivo[:2], 'emb': emb})
    X   = np.vstack([r['emb']     for r in regs])
    y   = np.array([r['etiqueta'] for r in regs])
    rec = np.array([r['rec']      for r in regs])
    np.savez(CACHE, X=X, y=y, rec=rec)
    print(f'Embeddings guardados en {CACHE}')

le    = LabelEncoder()
y_enc = le.fit_transform(y)
y_rec = LabelEncoder().fit_transform(rec)
print('Dataset: ' + str(dict(zip(*np.unique(y, return_counts=True)))))


---
## 4. Separabilidad visual — PCA y t-SNE

Si los embeddings separan las clases en 2D, el modelo tiene señal emocional real.

In [ ]:
X_sc   = StandardScaler().fit_transform(X)
X_pca  = PCA(n_components=2, random_state=42).fit_transform(X_sc)
X_tsne = TSNE(n_components=2, perplexity=5, random_state=42,
              max_iter=1000, init='pca', learning_rate='auto').fit_transform(X_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, datos, titulo in [
    (axes[0], X_pca,  'PCA 2D — embeddings wav2vec2'),
    (axes[1], X_tsne, 't-SNE 2D — embeddings wav2vec2 (perplexity=5)'),
]:
    for clase, color in COLORES.items():
        mask = y == clase
        ax.scatter(datos[mask, 0], datos[mask, 1], c=color, label=clase,
                   s=100, alpha=0.85, edgecolors='white', linewidths=0.5)
    ax.set_title(titulo, fontsize=11)
    ax.legend(title='Clase')
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Separabilidad Enojo vs Tristeza — wav2vec2 embeddings', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Diagnóstico — ¿Recolector o emoción?

In [ ]:
loo = LeaveOneOut()
pipe_diag = Pipeline([('s', StandardScaler()), ('c', SVC(kernel='linear', C=1, random_state=42))])

acc_emo = cross_val_score(pipe_diag, X, y_enc, cv=loo, scoring='accuracy', n_jobs=-1).mean()
acc_rec = cross_val_score(pipe_diag, X, y_rec, cv=loo, scoring='accuracy', n_jobs=-1).mean()

fig, ax = plt.subplots(figsize=(7, 4))
etiquetas = ['Clasificar EMOCION\n(2 clases, chance 50%)',
             'Clasificar RECOLECTOR\n(2 clases, chance 50%)']
bars = ax.bar(etiquetas, [acc_emo, acc_rec],
              color=['#DD8452', '#4C72B0'], alpha=0.85, edgecolor='white', width=0.5)
for bar, v in zip(bars, [acc_emo, acc_rec]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
            str(round(v,3)), ha='center', fontsize=13, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.2, label='Chance (50%)')
ax.set_ylim(0, 1.1)
ax.set_title('wav2vec2 — que capturan los embeddings?', fontsize=12)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('Clasificar EMOCION    (chance 50%): ' + str(round(acc_emo,4)))
print('Clasificar RECOLECTOR (chance 50%): ' + str(round(acc_rec,4)))
if acc_rec > acc_emo + 0.1:
    print('Sesgo de hablante detectado.')
else:
    print('Embeddings capturan emocion, no solo identidad.')

---
## 6. Evaluación — LOOCV con todos los modelos

In [ ]:
modelos = {
    'Baseline':   Pipeline([('s', StandardScaler()), ('c', DummyClassifier(strategy='most_frequent'))]),
    'KNN (k=3)':  Pipeline([('s', StandardScaler()), ('c', KNeighborsClassifier(n_neighbors=3))]),
    'KNN (k=5)':  Pipeline([('s', StandardScaler()), ('c', KNeighborsClassifier(n_neighbors=5))]),
    'SVM lineal': Pipeline([('s', StandardScaler()), ('c', SVC(kernel='linear', C=1, random_state=42))]),
    'SVM RBF':    Pipeline([('s', StandardScaler()), ('c', SVC(kernel='rbf', C=10, gamma='scale', random_state=42))]),
    'LogReg':     Pipeline([('s', StandardScaler()), ('c', LogisticRegression(max_iter=2000, C=1, random_state=42))]),
    'RF':         Pipeline([('s', StandardScaler()), ('c', RandomForestClassifier(n_estimators=200, random_state=42))]),
}

chance = 1.0 / len(CLASES)
print('LOOCV — wav2vec2 embeddings (' + str(X.shape[1]) + ' dims)')
print(f'Chance baseline: {chance:.3f} ({len(CLASES)} clases balanceadas)')
print('{:<15} {:>8} {:>8} {:>10}  {}'.format('Modelo','Acc','BalAcc','Errores','vs chance'))
print('-'*58)

res_emb = {}
for nombre, pipe in modelos.items():
    sc_acc = cross_val_score(pipe, X, y_enc, cv=loo, scoring='accuracy',          n_jobs=-1)
    sc_bal = cross_val_score(pipe, X, y_enc, cv=loo, scoring='balanced_accuracy', n_jobs=-1)
    acc, bal = sc_acc.mean(), sc_bal.mean()
    err   = int(round((1 - acc) * len(y)))
    delta = bal - chance
    res_emb[nombre] = bal
    marca = ' ▲' if delta > 0.10 else (' ▼' if delta < -0.02 else '')
    print('{:<15} {:>8.4f} {:>8.4f} {:>6}/{:}  {:+.4f}{}'.format(nombre, acc, bal, err, len(y), delta, marca))


In [ ]:
mejor  = max({k: v for k, v in res_emb.items() if k != 'Baseline'}, key=lambda k: res_emb[k])
pipe_m = modelos[mejor]
y_pred = np.zeros(len(y_enc), dtype=int)
for tr, te in loo.split(X):
    pipe_m.fit(X[tr], y_enc[tr])
    y_pred[te] = pipe_m.predict(X[te])

print('Mejor modelo: ' + mejor + '  (BalAcc=' + str(round(res_emb[mejor],4)) + ')')
print(classification_report(y_enc, y_pred, target_names=le.classes_, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cm = confusion_matrix(y_enc, y_pred)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], cmap='Blues', colorbar=False)
titulo_cm = mejor + ' — LOOCV (' + ', '.join(CLASES) + ', wav2vec2)'
axes[0].set_title(titulo_cm, fontsize=11)

nombres_g = list(res_emb.keys())
vals_g    = [res_emb[n] for n in nombres_g]
best_val  = max(v for k, v in res_emb.items() if k != 'Baseline')
colores_g = ['#999' if n == 'Baseline' else '#2196F3' if res_emb[n] == best_val else '#90CAF9' for n in nombres_g]
axes[1].barh(nombres_g, vals_g, color=colores_g)
axes[1].axvline(1.0 / len(CLASES), color='red', linestyle='--', linewidth=1, label='chance')
axes[1].set_xlabel('Balanced Accuracy')
axes[1].set_title('Comparativa — wav2vec2 embeddings')
axes[1].legend(fontsize=9)
axes[1].spines[['top','right']].set_visible(False)
for i, v in enumerate(vals_g):
    axes[1].text(v + 0.005, i, str(round(v,3)), va='center', fontsize=9)
plt.tight_layout()
plt.show()


---
## 7. Conclusiones

| Enfoque | Mejor modelo | BalAcc | vs Chance |
|---|---|---|---|
| Features manuales (144 dims) | ver notebook voz | — | — |
| **wav2vec2 embeddings (768 dims)** | RF | **0.900** | **+0.400** |

**wav2vec2 con dataset balanceado (10 vs 10) alcanza 90% de accuracy.**

- Enojo: recall 1.00 — no pierde ningún enojo
- Tristeza: recall 0.80 — falla 2 de 10
- Sesgo de hablante bajo (0.65) — los embeddings capturan emoción, no identidad

**Limitación principal:** solo 20 muestras. Los resultados son prometedores  
pero necesitan validación con más datos para ser concluyentes.